# KoHRM-Text-1.4B Colab T4 Smoke Test

This notebook is designed for a Google Colab T4 runtime. It checks the latest public Hugging Face export quickly without requiring full text generation support.

Current public artifact status:

- `AutoTokenizer` loading is supported.
- `config.json` and `model.safetensors` inspection is supported.
- Plain `AutoModelForCausalLM.generate()` is not supported yet because the public model repo does not yet ship the custom HRM-Text `trust_remote_code` modeling files.

한국어 요약: 이 노트북은 T4 Colab에서 최신 공개 가중치와 토크나이저가 정상 다운로드/로드되는지 빠르게 확인하는 용도입니다. 아직 표준 Transformers 생성 경로는 지원하지 않으므로, 생성 품질 평가용 노트북이 아니라 공개 artifact 검증용 노트북입니다.

## 1. Install dependencies

Use `hf_transfer` when available to speed up Hub downloads.

In [ ]:
!pip -q install -U huggingface_hub hf_transfer transformers safetensors accelerate torch

## 2. Runtime and download settings

`DOWNLOAD_WEIGHTS=True` downloads `model.safetensors` as well. Set it to `False` if you only want a very quick tokenizer/config test.

In [ ]:
import os
import json
import math
from pathlib import Path

import torch
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer
from safetensors import safe_open

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
DOWNLOAD_WEIGHTS = True

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print("gpu memory GiB:", round(free / 2**30, 2), "/", round(total / 2**30, 2))

info = HfApi().model_info(REPO_ID, revision=REVISION)
print("latest hub sha:", info.sha)

## 3. Download the latest public files

In [ ]:
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]
if DOWNLOAD_WEIGHTS:
    patterns.append("model.safetensors")

repo_dir = Path(snapshot_download(
    repo_id=REPO_ID,
    revision=REVISION,
    allow_patterns=patterns,
    max_workers=8,
))

print("downloaded to:", repo_dir)
print("files:")
for path in sorted(repo_dir.iterdir()):
    if path.is_file():
        print(" -", path.name, round(path.stat().st_size / 2**20, 2), "MiB")

## 4. Load config and tokenizer

In [ ]:
config = json.loads((repo_dir / "config.json").read_text())
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
}, indent=2, ensure_ascii=False))

tokenizer = AutoTokenizer.from_pretrained(repo_dir, use_fast=True)
print("tokenizer vocab size:", tokenizer.vocab_size)
print("bos/eos:", tokenizer.bos_token, tokenizer.eos_token)
print("special tokens:", tokenizer.all_special_tokens)

## 5. Tokenizer experiments

This checks Korean, terminal, JSON/tool-call, and code prompts with the project prompt wrapper.

In [ ]:
def format_prompt(prompt: str, condition_token: str = "<|object_ref_start|>") -> str:
    return f"<|im_start|>{condition_token}{prompt}<|im_end|>"

samples = {
    "korean_terminal": "한국어로 현재 디렉터리에서 용량이 큰 파일 20개를 찾고, .data 폴더는 별도로 합산하는 bash 명령을 작성하세요.",
    "tool_call_json": '{"tool":"exec_command","arguments":{"cmd":"du -h --max-depth=1 .data | sort -h","workdir":"/home/work"}}',
    "python_code": "def top_k_files(root, k=20):\n    return sorted(root.rglob('*'), key=lambda p: p.stat().st_size, reverse=True)[:k]",
    "english": "Explain why PrefixLM can use bidirectional attention over the input prefix while preserving causal generation over the response.",
}

rows = []
for name, text in samples.items():
    wrapped = format_prompt(text)
    ids = tokenizer(wrapped, add_special_tokens=False)["input_ids"]
    rows.append((name, len(text), len(ids), round(len(text) / max(1, len(ids)), 2), ids[:16]))

print(f"{'name':<18} {'chars':>8} {'tokens':>8} {'chars/token':>12} first_ids")
for row in rows:
    print(f"{row[0]:<18} {row[1]:>8} {row[2]:>8} {row[3]:>12} {row[4]}")

## 6. Inspect `model.safetensors` without loading all weights

This uses safetensors slices to read shapes from the file header. It is much faster and lighter than loading all tensors into CPU RAM.

In [ ]:
weights = repo_dir / "model.safetensors"
if not weights.exists():
    print("model.safetensors was not downloaded. Set DOWNLOAD_WEIGHTS=True and rerun cells 2-3.")
else:
    with safe_open(weights, framework="pt", device="cpu") as f:
        keys = list(f.keys())
        total_params = 0
        preview = []
        for key in keys:
            shape = tuple(f.get_slice(key).get_shape())
            n = math.prod(shape)
            total_params += n
            if len(preview) < 12:
                preview.append((key, shape, n))

        print("num tensors:", len(keys))
        print("num params:", f"{total_params:,}")
        print("bf16 weight size estimate GiB:", round(total_params * 2 / 2**30, 2))
        print("first tensors:")
        for key, shape, n in preview:
            print(" -", key, shape, f"{n:,}")

        first = keys[0]
        tensor = f.get_tensor(first)
        print("loaded one tensor:", first, tuple(tensor.shape), tensor.dtype, "mean=", float(tensor.float().mean()))

## 7. Expected Transformers generation status

This cell should fail gracefully today. The purpose is to confirm that the limitation is architecture support, not a broken download.

In [ ]:
try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(repo_dir, torch_dtype=torch.bfloat16, device_map="auto")
    print("Unexpected success: native Transformers model loaded.")
except Exception as exc:
    print("Expected for the current public export:")
    print(type(exc).__name__ + ":", str(exc)[:800])

## 8. What this notebook proves

- The latest Hub revision is visible from Colab.
- Tokenizer/config files are loadable.
- The 131K Korean/terminal tokenizer can encode project-style prompts.
- `model.safetensors` is structurally readable and has the expected parameter scale.

What it does not prove yet:

- It does not evaluate generation quality.
- It does not benchmark terminal/tool-call behavior.
- It does not replace the future `HrmTextForCausalLM` remote-code wrapper.